In [2]:
import { createAgent, tool } from "npm:langchain";

import { ChatGoogle } from "npm:@langchain/google";

import { z } from "npm:zod";

import { parse } from "jsr:@std/dotenv";


In [3]:
const env = parse(await Deno.readTextFile(".env"));

const GOOGLE_API_KEY = env.GOOGLE_API_KEY;

const API_URL = env.RESTAURANT_API_URL;


In [4]:
const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey: GOOGLE_API_KEY,
  temperature: 0,
});


In [ ]:
type RestaurantCategory =
  | "한식"
  | "일식"
  | "중식"
  | "양식"
  | "세계요리"
  | "특별한 술집"
  | "전통차/커피전문점"
  | "디저트/베이커리";

type Restaurant = {
  OPENDATA_ID: string;
  BZ_NM: string;
  GNG_CS: string;
  FD_CS: string;
  TLNO: string;
  MBZ_HR: string;
  SEAT_CNT: string;
  PKPL: string;
  HP: string;
  PSB_FRN: string;
  BKN_YN: string;
  INFN_FCL: string;
  BRFT_YN: string;
  DSSRT_YN: string;
  MNU: string;
  SMPL_DESC: string;
  SBW: string;
  BUS: string;
};

// 외부api 호출 함수
async function fetchRestaurants(district: string): Promise<Restaurant[]> {
  const url = new URL(API_URL);

  url.searchParams.set("addr", district);

  const response = await fetch(url);

  if (!response.ok) {
    throw new Error(`API 요청 실패: ${response.status}`);
  }

  const data = await response.json();

  // 실제 API 응답 구조에 맞게 수정
  return data.data as Restaurant[];
}

// 카테고리 필터 함수
function filterByCategory(
  restaurants: Restaurant[],
  category: RestaurantCategory | null,
): Restaurant[] {
  if (category === null) {
    return restaurants;
  }

  return restaurants.filter((restaurant) => restaurant.FD_CS === category);
}

// 지역,카테고리에 맞는 음식점 검색 함수
async function searchRestaurants(
  district: string,
  category: RestaurantCategory | null,
) {
  const restaurants = await fetchRestaurants(district);

  const filtered = filterByCategory(restaurants, category);

  return filtered;
}

const restaurantSearchTool = tool(
  // 실제로 실행할 함수
  async ({ district, category }) => {
    const restaurants = await searchRestaurants(district, category);

    return {
      count: restaurants.length,

      restaurants: restaurants.map((restaurant) => ({
        name: restaurant.BZ_NM,

        category: restaurant.FD_CS,

        address: restaurant.GNG_CS,

        menu: restaurant.MNU,

        parking: restaurant.PKPL,

        reservation: restaurant.BKN_YN,

        seats: restaurant.SEAT_CNT,

        description: restaurant.SMPL_DESC,
      })),
    };
  },
  // tool의 설명
  {
    name: "search_restaurants",

    description: `
대구광역시의 특정 지역에서
실제 음식점, 카페, 술집 정보를
공공데이터 API로 검색합니다.

사용자가 실제 음식점을 찾거나
추천을 요청할 때 사용합니다.

가격이 사용자의 예산에 맞는지
판정하는 용도로는 사용하지 마세요.
`,

    schema: z.object({
      district: z
        .string()
        .describe("검색할 대구광역시 행정구역. 예: 남구, 중구, 수성구"),

      category: z
        .enum([
          "한식",
          "일식",
          "중식",
          "양식",
          "세계요리",
          "특별한 술집",
          "전통차/커피전문점",
          "디저트/베이커리",
        ])
        .nullable()
        .describe("검색할 식당 카테고리. 특정 카테고리가 없으면 null"),
    }),
  },
);
